# 流式输出 Streaming

流式输出让 AI 的回复像打字一样逐字显示。

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

print("导入完成")

导入完成


## stream_mode="messages"——逐 Token 流式

最细粒度模式，每个 chunk 对应一个 Token。

In [2]:
model = init_chat_model("deepseek:deepseek-v4-flash")
agent = create_agent(model=model, system_prompt="你是菜鸟教程 RUNOOB 的助手。")

print("实时流式输出：")
for msg_chunk, metadata in agent.stream({"messages": [HumanMessage(content="用一句话介绍菜鸟教程 RUNOOB")]}, stream_mode="messages"):
    if msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

实时流式输出：
菜鸟教程 RUNOOB 是一个面向编程初学者的在线学习平台，提供丰富、易懂的编程语言和技术教程。


### 理解 metadata

metadata 包含 chunk 的来源信息。

In [5]:
print("查看 metadata 信息：\n")
for msg_chunk, metadata in agent.stream({"messages": [HumanMessage(content="你好")]}, stream_mode="messages"):
    if msg_chunk.content:
        print(f"内容: {msg_chunk.content}")
        print(f"来源节点: {metadata.get('langgraph_node')}")
        break

查看 metadata 信息：

内容: 你好
来源节点: model


## stream_mode="updates"——逐步查看 Agent 执行

构建显示"思考过程"的界面。

In [8]:
from langchain.tools import tool

@tool
def search_course(keyword: str) -> str:
    """搜索课程"""
    courses = {"python": "Python3 基础教程（30章，20小时）"}
    return courses.get(keyword.lower(), "未找到")

agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0), tools=[search_course], system_prompt="你是课程顾问。")

print("=== Agent 执行过程 ===\n")
for chunk in agent.stream({"messages": [HumanMessage(content="查一下 Python 课程")]}, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"[{node_name}]", end=" ")
        if "messages" in update:
            for msg in update["messages"]:
                if msg.type == "ai" and hasattr(msg, 'tool_calls') and msg.tool_calls:
                    print(f"请求调用: {[tc['name'] for tc in msg.tool_calls]}")
                elif msg.type == "tool":
                    print(f"工具返回 [{msg.name}]: {msg.content}")
                elif msg.type == "ai" and msg.content:
                    print(f"回复: {msg.content[:80]}")

=== Agent 执行过程 ===

[model] 请求调用: ['search_course']
[tools] 工具返回 [search_course]: Python3 基础教程（30章，20小时）
[model] 回复: 为您找到了以下 Python 课程：

---

🐍 **Python3 基础教程**
- **章节数**：30 章
- **时长**：20 小时
- **内容


## stream_mode="custom"——自定义事件

Middleware 通过 `runtime.stream_writer()` 发送。

In [11]:
from langchain.agents.middleware import before_model, after_model

@before_model
def notify_before(state, runtime):
    """模型调用前发送事件"""
    runtime.stream_writer({"type": "status", "message": "正在思考..."})
    return None

@after_model
def notify_after(state, runtime):
    """模型调用后发送事件"""
    last_msg = state["messages"][-1] if state.get("messages") else None
    has_tools = hasattr(last_msg, "tool_calls") and last_msg.tool_calls
    msg = "正在调用工具..." if has_tools else "回答已完成"
    runtime.stream_writer({"type": "status", "message": msg})
    return None

agent = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0),
    tools=[search_course],
    middleware=[notify_before, notify_after],
    system_prompt="你是课程顾问。",
)

for mode, chunk in agent.stream(
    {"messages": [HumanMessage(content="查一下 Python 课程")]},
    stream_mode=["updates", "custom"],
):
    if mode == "custom":
        print(f"[事件] {chunk['message']}")


[事件] 正在思考...
[事件] 正在调用工具...
[事件] 正在思考...
[事件] 回答已完成


## 异步流式输出

Web 服务中使用 `astream()`。

In [12]:
import asyncio

async def stream_agent():
    agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash"), system_prompt="你是助手。")
    full = ""
    async for msg_chunk, metadata in agent.astream({"messages": [HumanMessage(content="一句话介绍菜鸟教程")]}, stream_mode="messages"):
        if msg_chunk.content:
            full += msg_chunk.content
            print(msg_chunk.content, end="", flush=True)
    print(f"\n\n长度: {len(full)} 字")

await stream_agent()
print("取消注释 await stream_agent() 运行")

菜鸟教程是一个面向编程初学者的在线学习平台，提供简洁易懂的各类技术教程与在线工具。

长度: 41 字
取消注释 await stream_agent() 运行


**术语：** AIMessageChunk、stream_mode、stream_writer()、astream()